# 23b — Pengujian Performa dan Kebutuhan Sistem QA RAG

Ukur waktu respons, CPU, RAM, dan VRAM proses QA, kemudian hitung kebutuhan layanan berdasarkan skenario penggunaan. Restart kernel, pastikan backend dan Ollama berjalan, lalu jalankan semua sel dari atas. Sel pertama memasang dependensi yang belum tersedia. Hasil tiap sesi disimpan dalam folder baru.

In [4]:
import sys
import subprocess
import importlib.util
import importlib.metadata

packages = {
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy",
    "psutil": "psutil",
    "IPython": "ipython",
    "pynvml": "nvidia-ml-py",
}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
try:
    importlib.metadata.version("nvidia-ml-py")
except importlib.metadata.PackageNotFoundError:
    if "nvidia-ml-py" not in missing:
        missing.append("nvidia-ml-py")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

import pynvml
print("Dependensi siap. Python:", sys.executable)
print("Pembaca VRAM:", importlib.metadata.version("nvidia-ml-py"))

Dependensi siap. Python: d:\Laptop\Anaconda\envs\rag-sistem\python.exe
Pembaca VRAM: 13.615.71


In [5]:
import os
import sys
import json
import math
import time
import uuid
import hashlib
import platform
import threading
import subprocess
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone, timedelta
from urllib.parse import urlparse

import requests
import pandas as pd
import numpy as np
import psutil
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("QA_PROJECT_ROOT",
    r"D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\Sistem QA RAG"))
RESEARCH_ROOT = Path(os.environ.get("QA_RESEARCH_ROOT",
    r"D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil"))
BASE_URL = os.environ.get("RAG_API_URL", "http://127.0.0.1:8000").rstrip("/")
OLLAMA_URL = os.environ.get("INFERENCE_URL", "http://127.0.0.1:11434").rstrip("/")
OUTPUT_BASE = Path(os.environ.get("SYSTEM_EVAL_OUTPUT",
    str(RESEARCH_ROOT / "Hasil" / "tujuan3" / "versi-final")))

MODEL_KEYS = ["llama3.2-finetuned", "gemma2-finetuned"]
TEST_QUESTIONS = [
    "Berapa persentase penduduk lanjut usia di Indonesia menurut publikasi Statistik Penduduk Lanjut Usia edisi terbaru?",
    "Apa yang dimaksud dengan Indeks Khusus Penanganan Stunting (IKPS) menurut publikasi BPS?",
    "Bagaimana tren Indeks Pembangunan Manusia (IPM) Indonesia dari edisi ke edisi?",
    "Berapa Indeks Perilaku Anti Korupsi (IPAK) nasional pada edisi terakhir?",
]
RESPONSE_STYLE = "detail"
TOP_K = 5
N_REPEAT = 3
N_WARMUP = 1  # Pertanyaan pertama per model; tidak masuk 24 baris terukur.
REQUEST_TIMEOUT = (10, 900)
SAMPLE_INTERVAL_S = 0.05
MEASURE_LOCAL_RESOURCES = True
BACKEND_PID = None  # Isi PID listener backend jika deteksi otomatis tidak tersedia.
OLLAMA_PID = None   # Isi PID layanan Ollama jika deteksi otomatis tidak tersedia.
ISOLATED_RUN_CONFIRMED = True  # True hanya jika backend/Ollama tidak dipakai klien lain.
RUN_CAPACITY_PROFILE = True # True: tambahan 72 permintaan terukur + pemanasan.
CAPACITY_CONCURRENCY_LEVELS = [1, 2, 4]
CAPACITY_REPEATS = 3
IDLE_SAMPLE_SECONDS = 2.0
HARDWARE_NOTES = ""  # Contoh: mode daya, aplikasi lain, perubahan perangkat.
BACKEND_CONFIGURATION_NOTES = ""  # Misal workers=1, concurrency=2, backend sudah direstart.

assert RESPONSE_STYLE in {"ringkas", "detail"}
assert type(N_REPEAT) is int and N_REPEAT >= 1
assert type(N_WARMUP) is int and N_WARMUP >= 0
assert type(TOP_K) is int and 1 <= TOP_K <= 5
assert len(MODEL_KEYS) == len(set(MODEL_KEYS)) and MODEL_KEYS
assert TEST_QUESTIONS and all(isinstance(q, str) and q.strip() for q in TEST_QUESTIONS)
assert SAMPLE_INTERVAL_S >= 0.02
assert type(ISOLATED_RUN_CONFIRMED) is bool and type(RUN_CAPACITY_PROFILE) is bool
assert all(type(c) is int and 1 <= c <= 8 for c in CAPACITY_CONCURRENCY_LEVELS)
assert len(CAPACITY_CONCURRENCY_LEVELS) == len(set(CAPACITY_CONCURRENCY_LEVELS))
assert type(CAPACITY_REPEATS) is int and CAPACITY_REPEATS >= 1
assert IDLE_SAMPLE_SECONDS >= 1
if MEASURE_LOCAL_RESOURCES:
    for address in (BASE_URL, OLLAMA_URL):
        if urlparse(address).hostname not in {"localhost", "127.0.0.1", "::1"}:
            raise ValueError("Sampling lokal hanya untuk backend/Ollama lokal. Untuk server lain, matikan MEASURE_LOCAL_RESOURCES; memori server tidak akan terukur.")
if not (PROJECT_ROOT / "app" / "pipeline.py").is_file():
    raise FileNotFoundError("QA_PROJECT_ROOT tidak mengarah ke folder sistem.")

RUN_ID = datetime.now(timezone(timedelta(hours=7))).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8]
OUT = OUTPUT_BASE / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
(OUT / "responses").mkdir()
(OUT / "resources").mkdir()
METRICS_CSV = OUT / "system_performance_all.csv"
ROWS = []
print("Folder hasil:", OUT)
print("Permintaan terukur:", len(MODEL_KEYS) * len(TEST_QUESTIONS) * N_REPEAT)
print("Mode: API produksi, RAG wajib, cache mengikuti sistem.")

Folder hasil: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil\Hasil\tujuan3\versi-final\20261012-200440-99c5f27b
Permintaan terukur: 24
Mode: API produksi, RAG wajib, cache mengikuti sistem.


## 1. Kesiapan layanan

Periksa API, model, dan konfigurasi. Metadata lingkungan disimpan untuk mencatat kondisi pengujian; spesifikasi laptop hanya menjadi identitas perangkat uji.

In [6]:
def save_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def json_get(url):
    response = requests.get(url, timeout=(5, 30))
    response.raise_for_status()
    return response.json()

health = json_get(BASE_URL + "/api/health")
ready = json_get(BASE_URL + "/api/ready")
if not ready.get("ready") or not health.get("inference_reachable"):
    raise RuntimeError("Backend atau Ollama belum siap.")
missing_models = set(MODEL_KEYS) - set(health.get("available_models", []))
if missing_models:
    raise RuntimeError("Model belum tersedia: " + ", ".join(sorted(missing_models)))
openapi = json_get(BASE_URL + "/openapi.json")
if "/api/ask" not in openapi.get("paths", {}):
    raise RuntimeError("Endpoint pengujian tidak tersedia.")

try:
    model_inventory = [{"name": m.get("name"), "digest": m.get("digest"),
                        "size": m.get("size"), "details": m.get("details")}
                       for m in json_get(OLLAMA_URL + "/api/tags").get("models", [])
                       if str(m.get("name", "")).removesuffix(":latest") in MODEL_KEYS]
except requests.RequestException as exc:
    model_inventory = {"unavailable": type(exc).__name__}

SAFE_KEYS = {
    "QUEUE_CONCURRENCY", "MAX_QUEUE", "QUEUE_WAIT_TIMEOUT", "CACHE_TTL",
    "OLLAMA_NUM_CTX", "OLLAMA_NUM_BATCH", "OLLAMA_NUM_GPU", "OLLAMA_KEEP_ALIVE",
    "OLLAMA_TIMEOUT", "MAX_NEW_TOKENS", "EMBEDDING_DEVICE", "RERANKER_DEVICE",
    "RERANK_SCORE_THRESHOLD", "KB_REVISION", "EMBEDDING_REVISION", "RERANKER_REVISION",
}
local_env_settings = {}
env_file = PROJECT_ROOT / ".env"
if env_file.is_file():
    for line in env_file.read_text(encoding="utf-8-sig").splitlines():
        key, sep, value = line.partition("=")
        if sep and key.strip() in SAFE_KEYS:
            local_env_settings[key.strip()] = value.strip().strip('"').strip("'")
source_files = sorted((PROJECT_ROOT / "app").glob("*.py")) + [PROJECT_ROOT / "web" / "question-rules.json"]
ENVIRONMENT = {
    "run_id": RUN_ID, "created_at": datetime.now(timezone(timedelta(hours=7))).isoformat(),
    "system_variant": "rag_excerpt_only_sentence_completion",
    "endpoint": "/api/ask", "backend_url": BASE_URL, "ollama_url": OLLAMA_URL,
    "api_version": openapi.get("info", {}).get("version"),
    "local_source_sha256": {str(p.relative_to(PROJECT_ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
                            for p in source_files if p.is_file()},
    "source_hash_note": "Hash kode lokal; proses backend perlu direstart agar sesuai. Versi API saja tidak membuktikan versi kode.",
    "local_env_settings_not_verified_runtime": local_env_settings,
    "backend_configuration_notes": BACKEND_CONFIGURATION_NOTES,
    "model_inventory_before": model_inventory, "model_keys": MODEL_KEYS,
    "test_questions": TEST_QUESTIONS, "response_style": RESPONSE_STYLE, "top_k": TOP_K,
    "repeat_per_question": N_REPEAT, "warmup_per_model": N_WARMUP,
    "request_order": "model -> repeat -> question; sequential",
    "cache_policy": "Production cache unchanged; warmup and every measured request may populate cache. No cold-start claim.",
    "sample_interval_s": SAMPLE_INTERVAL_S, "memory_unit": "MiB",
    "resource_measurement_enabled": MEASURE_LOCAL_RESOURCES,
    "package_versions": {name: importlib.metadata.version(name) for name in ("requests", "pandas", "numpy", "psutil")},
    "metric_definitions": {
        "retrieval_s": "backend expansion_s + search_rerank_s (includes context fitting/metadata work timed in that stage)",
        "generation_s": "backend answer stream until model stop or application stop; not decode-only",
        "citation_s": "Must be zero: source summaries removed; numeric references still exist in main answer",
        "backend_total_s": "backend queue + processing",
        "client_total_s": "wall time for POST and complete JSON response; excludes resource sampler setup/teardown and file saving",
        "first_token_s": "backend time to first raw draft token, including queue; not browser/client TTFT",
        "process_rss_mb": "sum of RSS for unique backend + Ollama process-tree PIDs; shared pages may be counted more than once",
        "gpu_mb": "total used VRAM of entire NVIDIA devices; not per-request allocation",
        "status_ok": "HTTP/response contract success, NOT answer correctness",
    },
    "hardware_notes": HARDWARE_NOTES,
}
save_json(OUT / "system_performance_environment.json", ENVIRONMENT)
print("Backend siap. Model:", ", ".join(MODEL_KEYS))


Backend siap. Model: llama3.2-finetuned, gemma2-finetuned


## 2. Pengukuran sumber daya

CPU dan RAM diukur dari proses backend, Ollama, dan turunannya. RSS dapat menghitung memori bersama lebih dari sekali. VRAM dibaca per proses dan per GPU; jika driver tidak mendukungnya, nilai dibiarkan kosong.

Pastikan tidak ada klien lain memakai layanan selama pengujian. Interval sampling ditetapkan 50 ms; puncak sesaat dapat terlewat.

In [7]:
MIB = 1024 ** 2
RESOURCE_SCHEMA = "qa_process_v2"
RESOURCE_NOTES, GPU_HANDLES = [], []
pynvml = None
if MEASURE_LOCAL_RESOURCES:
    try:
        import pynvml
        pynvml.nvmlInit()
        GPU_HANDLES = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(pynvml.nvmlDeviceGetCount())]
    except Exception as exc:
        RESOURCE_NOTES.append("VRAM per proses tidak tersedia: " + type(exc).__name__)

def listener_pid(url, configured_pid=None):
    if configured_pid is not None:
        p = psutil.Process(int(configured_pid))
        return (p.pid, p.create_time())
    port = urlparse(url).port or (443 if urlparse(url).scheme == "https" else 80)
    try:
        ids = {c.pid for c in psutil.net_connections(kind="tcp")
               if c.status == psutil.CONN_LISTEN and c.laddr.port == port and c.pid}
        if len(ids) == 1:
            p = psutil.Process(ids.pop())
            return (p.pid, p.create_time())
    except (psutil.Error, OSError) as exc:
        RESOURCE_NOTES.append("Deteksi PID gagal: " + type(exc).__name__)
    return None

BACKEND_ROOT = listener_pid(BASE_URL, BACKEND_PID) if MEASURE_LOCAL_RESOURCES else None
OLLAMA_ROOT = listener_pid(OLLAMA_URL, OLLAMA_PID) if MEASURE_LOCAL_RESOURCES else None

def process_tree_stats(identity):
    if identity is None:
        return None
    try:
        root = psutil.Process(identity[0])
        if root.create_time() != identity[1]:
            return None
        records = {}
        for p in [root] + root.children(recursive=True):
            created = p.create_time()
            cpu = p.cpu_times()
            records[f"{p.pid}:{created}"] = {
                "pid": p.pid, "created": created, "rss_mb": p.memory_info().rss / MIB,
                "cpu_s": cpu.user + cpu.system,
            }
        return records
    except (psutil.Error, OSError):
        return None  # Jangan melaporkan pohon parsial sebagai data lengkap.

def gpu_process_snapshot(processes):
    result = {}
    if processes is None:
        return result
    selected = {p["pid"] for p in processes.values()}
    for index, handle in enumerate(GPU_HANDLES):
        key = str(index)
        try:
            # PID yang muncul di kedua daftar dihitung sekali, bukan dijumlahkan.
            entries = list(pynvml.nvmlDeviceGetComputeRunningProcesses(handle))
            entries += list(pynvml.nvmlDeviceGetGraphicsRunningProcesses(handle))
            by_pid = {}
            unknown = False
            for item in entries:
                if item.pid not in selected:
                    continue
                value = getattr(item, "usedGpuMemory", None)
                if not isinstance(value, int) or value < 0 or value >= (2**64 - 1):
                    unknown = True
                    continue
                by_pid[item.pid] = max(by_pid.get(item.pid, 0), value)
            result[key] = {
                "qa_vram_mb": None if unknown else sum(by_pid.values()) / MIB,
                "status": "unavailable_per_pid" if unknown else "measured",
                "qa_pids": sorted(by_pid),
            }
        except Exception as exc:
            result[key] = {"qa_vram_mb": None, "status": type(exc).__name__, "qa_pids": []}
    return result

def resource_snapshot():
    if not MEASURE_LOCAL_RESOURCES:
        return {"at": time.perf_counter(), "epoch": time.time(), "processes": None, "gpus": {}}
    backend = process_tree_stats(BACKEND_ROOT)
    ollama = process_tree_stats(OLLAMA_ROOT)
    combined = {**backend, **ollama} if backend is not None and ollama is not None else None
    return {
        "at": time.perf_counter(), "epoch": time.time(),
        "backend_rss_mb": None if backend is None else sum(p["rss_mb"] for p in backend.values()),
        "ollama_rss_mb": None if ollama is None else sum(p["rss_mb"] for p in ollama.values()),
        "process_rss_mb": None if combined is None else sum(p["rss_mb"] for p in combined.values()),
        "processes": combined, "gpus": gpu_process_snapshot(combined),
    }

def process_cpu_delta(previous, current):
    old, new = previous.get("processes"), current.get("processes")
    if old is None or new is None or set(old) - set(new):
        return None  # Proses hilang/restart: ujung counter tidak diketahui.
    total = 0.0
    for key, p in new.items():
        if key in old:
            delta = p["cpu_s"] - old[key]["cpu_s"]
        elif p["created"] >= previous["epoch"]:
            delta = p["cpu_s"]  # Proses lahir di dalam interval.
        else:
            return None
        if delta < 0:
            return None
        total += delta
    return total

class ResourceSampler:
    def __init__(self):
        self.stop_event = threading.Event()
        self.samples, self.failures = [], 0

    def capture(self):
        try:
            self.samples.append(resource_snapshot())
        except Exception:
            self.failures += 1

    def run(self):
        while not self.stop_event.wait(SAMPLE_INTERVAL_S):
            self.capture()

    def __enter__(self):
        self.capture()
        self.thread = threading.Thread(target=self.run, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, *args):
        self.stop_event.set()
        self.thread.join()
        self.capture()

    def as_dict(self):
        result = {"resource_schema": RESOURCE_SCHEMA,
                  "resource_scope": "backend_and_ollama_unique_process_trees",
                  "isolated_run_confirmed": ISOLATED_RUN_CONFIRMED,
                  "resource_samples": len(self.samples), "resource_sample_errors": self.failures,
                  "sample_interval_s": SAMPLE_INTERVAL_S, "memory_unit": "MiB"}
        for key in ("backend_rss_mb", "ollama_rss_mb", "process_rss_mb"):
            values = [s[key] for s in self.samples if s.get(key) is not None]
            base = self.samples[0].get(key) if self.samples else None
            peak = max(values) if values else None
            result["baseline_" + key] = base
            result["peak_" + key] = peak
            result["valid_samples_" + key] = len(values)
            result["delta_" + key] = None if base is None or peak is None else peak - base
        deltas, rates, gaps = [], [], []
        for a, b in zip(self.samples, self.samples[1:]):
            dt = b["at"] - a["at"]
            delta = process_cpu_delta(a, b)
            gaps.append(dt)
            if delta is not None and dt > 0:
                deltas.append(delta)
                rates.append(delta / dt)
        complete = len(self.samples) >= 2 and len(deltas) == len(self.samples)-1 and self.failures == 0
        wall = self.samples[-1]["at"] - self.samples[0]["at"] if len(self.samples) >= 2 else None
        result.update(
            qa_cpu_counter_complete=complete,
            qa_cpu_core_seconds=sum(deltas) if complete else None,
            qa_cpu_observed_partial_core_seconds=sum(deltas) if deltas else None,
            qa_cpu_mean_cores=sum(deltas)/wall if complete and wall and wall > 0 else None,
            qa_cpu_peak_cores=max(rates) if rates else None,
            resource_window_s=wall, actual_sample_gap_max_s=max(gaps) if gaps else None)
        device_peaks = {}
        for key in sorted({k for s in self.samples for k in s["gpus"]}):
            readings = [s["gpus"].get(key, {}).get("qa_vram_mb") for s in self.samples]
            values = [v for v in readings if v is not None]
            device_peaks[key] = {
                "peak_qa_vram_mb": max(values) if values else None,
                "valid_samples": len(values), "complete": bool(values) and len(values) == len(self.samples),
                "statuses": sorted({s["gpus"].get(key, {}).get("status", "missing") for s in self.samples}),
            }
        result["qa_gpu_devices_json"] = json.dumps(device_peaks, ensure_ascii=False)
        complete_gpu = bool(device_peaks) and all(v["complete"] for v in device_peaks.values()) and self.failures == 0
        result["qa_gpu_sampling_complete"] = complete_gpu
        # Peak per perangkat; VRAM beberapa GPU tidak dianggap satu kartu besar.
        result["peak_qa_gpu_single_device_mb"] = max(v["peak_qa_vram_mb"] for v in device_peaks.values()) if complete_gpu else None
        return result

gpu_inventory = []
for index, handle in enumerate(GPU_HANDLES):
    name = pynvml.nvmlDeviceGetName(handle)
    gpu_inventory.append({"index": index, "name": name.decode() if isinstance(name, bytes) else name,
                          "total_vram_mib": pynvml.nvmlDeviceGetMemoryInfo(handle).total / MIB})
ENVIRONMENT["resource_schema"] = RESOURCE_SCHEMA
ENVIRONMENT["isolated_run_confirmed"] = ISOLATED_RUN_CONFIRMED
ENVIRONMENT["hardware_inventory"] = {
    "purpose": "identitas alat ukur; BUKAN rekomendasi kapasitas server",
    "os": platform.platform(), "python": platform.python_version(),
    "cpu": platform.processor(), "physical_cores": psutil.cpu_count(logical=False),
    "logical_cores": psutil.cpu_count(logical=True), "total_ram_mib": psutil.virtual_memory().total / MIB,
    "nvidia_gpus_sampled": gpu_inventory,
}
ENVIRONMENT["metric_definitions"].update({
    "qa_cpu_core_seconds": "sum of user+system CPU time deltas for unique QA PIDs; sample window includes small client/sampler overhead",
    "qa_cpu_mean_cores": "QA core-seconds / sampled wall seconds; 1 means one logical CPU continuously occupied, not 1% of laptop",
    "peak_qa_gpu_single_device_mb": "maximum per-device VRAM attributed to QA PIDs, no device-wide fallback",
    "process_rss_mb": "sum RSS unique backend/Ollama PIDs; conservative resident footprint, shared pages can be counted more than once",
})
ENVIRONMENT["metric_definitions"].pop("gpu_mb", None)
ENVIRONMENT["resource_process_roots"] = {"backend": BACKEND_ROOT, "ollama": OLLAMA_ROOT}
ENVIRONMENT["resource_notes"] = RESOURCE_NOTES + [
    "Do not run other clients against the same backend/Ollama during a measurement.",
    "Short-lived processes and peaks between samples may be missed.",
    "GPU PID memory may be unavailable on Windows WDDM; missing is not zero.",
    "Notebook/client CPU and host-wide CPU/RAM/GPU usage are excluded from resource estimates.",
]
save_json(OUT / "system_performance_environment.json", ENVIRONMENT)
print("PID backend:", BACKEND_ROOT, "| PID Ollama:", OLLAMA_ROOT)
print("Pengukuran proses QA saja. Isolasi dikonfirmasi:", ISOLATED_RUN_CONFIRMED)
for note in RESOURCE_NOTES:
    print(note)

PID backend: (2828, 1791810041.25376) | PID Ollama: (17640, 1791747793.562884)
Pengukuran proses QA saja. Isolasi dikonfirmasi: True


## 3. Validasi dan penyimpanan

Respons harus memakai RAG dan cuplikan sumber tanpa generasi ringkasan sitasi. Status jawaban dan metrik token yang tersedia dicatat apa adanya. Jika CSV terkunci, hasil dialihkan ke file pengganti agar pengujian tetap berjalan.

In [8]:
def finite_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)

def validate_final_response(body, model_key):
    required = {"answer", "answer_status", "sources", "latency", "config", "cache",
                "generation_metrics", "process", "warnings", "request_id"}
    if not isinstance(body, dict) or required - set(body):
        raise ValueError("Kontrak respons tidak lengkap.")
    cfg, latency = body["config"], body["latency"]
    if cfg.get("model_key") != model_key or cfg.get("use_rag") is not True or cfg.get("evaluation") is not False:
        raise ValueError("Model/mode respons tidak sama dengan API produksi yang diminta.")
    if cfg.get("response_style") != RESPONSE_STYLE:
        raise ValueError("Panjang jawaban respons tidak sesuai permintaan.")
    if not isinstance(body["answer"], str) or not body["answer"].strip():
        raise ValueError("Jawaban kosong.")
    if body["answer_status"] not in {"needs_review", "basic_checks_passed", "no_evidence"}:
        raise ValueError("Status jawaban tidak sesuai versi akhir.")
    for key in ("expansion_s", "search_rerank_s", "retrieval_s", "generation_s", "citation_s",
                "processing_s", "queue_s", "total_s"):
        if not finite_number(latency.get(key)) or latency[key] < 0:
            raise ValueError("Metrik waktu tidak valid: " + key)
    if latency.get("first_token_s") is not None and (not finite_number(latency["first_token_s"]) or latency["first_token_s"] < 0):
        raise ValueError("Waktu token pertama tidak valid.")
    if latency["citation_s"] != 0:
        raise ValueError("Backend masih menjalankan generasi ringkasan sumber.")
    if abs(latency["retrieval_s"] - latency["expansion_s"] - latency["search_rerank_s"]) > 0.01:
        raise ValueError("Definisi retrieval tidak sesuai.")
    if abs(latency["total_s"] - latency["processing_s"] - latency["queue_s"]) > 0.02:
        raise ValueError("Definisi waktu total tidak sesuai.")
    if not isinstance(body["sources"], list) or len(body["sources"]) > TOP_K:
        raise ValueError("Daftar sumber tidak sesuai.")
    for source in body["sources"]:
        if source.get("answer") != "" or source.get("citation_status") != "excerpt_only":
            raise ValueError("Sumber masih memakai ringkasan model.")
        quote, text = source.get("quote"), source.get("source_text")
        if not isinstance(quote, str) or not isinstance(text, str) or not quote.strip() or quote not in text:
            raise ValueError("Cuplikan bukan potongan literal source_text.")
    if any(type(body["cache"].get(k)) is not bool for k in ("enabled", "expansion", "search_rerank")):
        raise ValueError("Status cache tidak lengkap.")
    return body

def flatten_response(body):
    latency, cfg, metrics = body["latency"], body["config"], body["generation_metrics"]
    cache, process = body["cache"], body["process"]
    provider_available = metrics.get("provider_metrics_available") is True
    count = metrics.get("eval_count") if provider_available else None
    duration_ns = metrics.get("eval_duration") if provider_available else None
    tokens_per_s = (count / (duration_ns / 1e9)
                   if finite_number(count) and finite_number(duration_ns) and duration_ns > 0 else None)
    complete_sources = body["sources"]
    answer = body["answer"]
    row = {
        **{k: latency.get(k) for k in ("expansion_s", "search_rerank_s", "retrieval_s", "generation_s",
                                      "citation_s", "first_token_s", "processing_s", "queue_s")},
        "backend_total_s": latency.get("total_s"), "answer": answer, "answer_chars": len(answer),
        "answer_status": body["answer_status"], "request_id": body["request_id"],
        "source_count": len(complete_sources),
        "source_summaries_generated": False,
        "fallback_excerpt_answer": answer.startswith("Bagian sumber yang paling dekat dengan pertanyaan:"),
        "sources_json": json.dumps(complete_sources, ensure_ascii=False),
        "warnings_json": json.dumps(body["warnings"], ensure_ascii=False),
        "expanded_queries_json": json.dumps(process.get("expanded_queries", []), ensure_ascii=False),
        "expansion_origin": process.get("expansion_origin"),
        "expansion_note": process.get("expansion_note"),
        "candidates_before_rerank": process.get("candidates_before_rerank"),
        "cache_enabled": cache.get("enabled"), "cache_expansion": cache.get("expansion"),
        "cache_search_rerank": cache.get("search_rerank"),
        "provider_metrics_available": provider_available,
        "output_finish_reason": metrics.get("output_finish_reason"),
        "stopped_early": metrics.get("stopped_early"),
        "trimmed_incomplete_sentence": metrics.get("trimmed_incomplete_sentence"),
        "soft_character_limit": metrics.get("soft_character_limit"),
        "answer_tokens_per_s": tokens_per_s,
        "ollama_eval_count": count,
        **{k: cfg.get(k) for k in ("num_ctx", "num_batch", "num_gpu", "threshold", "kb_fingerprint")},
    }
    for key in ("load_duration", "prompt_eval_count", "prompt_eval_duration", "eval_duration",
                "total_duration", "done_reason"):
        row["ollama_" + key] = metrics.get(key) if provider_available else None
    return row

def request_payload(question, model):
    return {"question": question, "model_key": model, "use_rag": True,
            "top_k": TOP_K, "response_style": RESPONSE_STYLE}

LEADING_COLUMNS = [
    "run_id", "test_scope", "model_key", "question_id", "question", "repeat", "status", "status_code",
    "answer_status", "retrieval_s", "generation_s", "backend_total_s", "client_total_s",
    "peak_backend_rss_mb", "peak_ollama_rss_mb", "peak_process_rss_mb", "peak_qa_gpu_single_device_mb", "qa_cpu_core_seconds", "qa_cpu_mean_cores",
    "cache_expansion", "cache_search_rerank", "source_count", "answer", "citation_s",
]
def resolve_latest_csv(path, logical_name):
    """Read the active CSV pointer; never follow a path outside the result folder."""
    path = Path(path)
    if path.name != logical_name:
        return path  # Path hasil pengganti yang dipilih eksplisit harus dihormati.
    pointer = path.parent / (logical_name + ".latest.json")
    if not pointer.is_file():
        return path
    info = json.loads(pointer.read_text(encoding="utf-8"))
    name = info.get("filename")
    stem = Path(logical_name).stem
    if (info.get("logical_name") != logical_name or not isinstance(name, str)
            or Path(name).name != name or "/" in name or "\\" in name
            or not name.endswith(".csv") or not (name == logical_name or name.startswith(stem + "-unlocked-"))):
        raise ValueError("Penunjuk CSV tidak valid: " + str(pointer))
    candidate = path.parent / name
    if not candidate.is_file():
        raise FileNotFoundError("CSV aktif belum ditemukan: " + str(candidate))
    return candidate

def save_csv_resilient(frame, path, logical_name):
    """Only a locked destination is recoverable; full disk/access failures still stop."""
    path = Path(path)
    temporary = path.parent / (logical_name + "." + uuid.uuid4().hex + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8-sig")
    actual = path
    try:
        temporary.replace(actual)
    except PermissionError:
        actual = path.parent / (Path(logical_name).stem + "-unlocked-" + uuid.uuid4().hex[:12] + ".csv")
        temporary.replace(actual)
        print("CSV sebelumnya terkunci/tidak dapat diganti. Hasil lengkap dilanjutkan ke:", actual)
    pointer = path.parent / (logical_name + ".latest.json")
    pointer_tmp = pointer.with_name(pointer.name + "." + uuid.uuid4().hex + ".tmp")
    info = {"logical_name": logical_name, "filename": actual.name,
            "row_count": len(frame), "updated_at_utc": datetime.now(timezone.utc).isoformat()}
    try:
        pointer_tmp.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
        pointer_tmp.replace(pointer)
    except PermissionError:
        print("CSV tersimpan, tetapi penunjuk hasil belum diperbarui. Gunakan path ini secara eksplisit:", actual)
    return actual

def flush_benchmark_results():
    global METRICS_CSV
    if not ROWS:
        raise RuntimeError("Tidak ada baris hasil dalam kernel ini.")
    data = pd.DataFrame(ROWS)
    columns = LEADING_COLUMNS + sorted(set(data.columns) - set(LEADING_COLUMNS))
    METRICS_CSV = save_csv_resilient(data.reindex(columns=columns), METRICS_CSV, "system_performance_all.csv")
    return METRICS_CSV

def record_row(row):
    key = (row["model_key"], row["question_id"], row["repeat"])
    if any((r["model_key"], r["question_id"], r["repeat"]) == key for r in ROWS):
        raise RuntimeError("Baris sudah tercatat; tidak ditambahkan dua kali.")
    ROWS.append(row)
    flush_benchmark_results()

def make_row(model, question_index, question, repeat):
    return dict(run_id=RUN_ID, test_scope="e2e_system_final", model_key=model,
                question_id=f"Q{question_index:02d}", question=question, repeat=repeat,
                response_style=RESPONSE_STYLE, top_k=TOP_K, use_rag=True,
                status="error", status_code=None)

## 4. Pengujian utama

Empat pertanyaan diuji pada dua model, masing-masing tiga kali: 24 permintaan terukur. Pemanasan tidak masuk perhitungan. Timeout atau kegagalan kontrak menghentikan sesi; hasil yang sudah terkumpul tetap disimpan.

In [9]:
def run_benchmark():
    if ROWS or METRICS_CSV.exists():
        raise RuntimeError("Sesi ini sudah berisi hasil. Jalankan ulang sel konfigurasi untuk folder sesi baru.")
    for model in MODEL_KEYS:
        for warmup in range(N_WARMUP):
            q = TEST_QUESTIONS[warmup % len(TEST_QUESTIONS)]
            response = requests.post(BASE_URL + "/api/ask", json=request_payload(q, model),
                                     timeout=REQUEST_TIMEOUT)
            try:
                response.raise_for_status()
                body = response.json()
            finally:
                response.close()
            save_json(OUT / "responses" / f"warmup_{model}_{warmup+1}.json", body)
            validate_final_response(body, model)
            print(model, "warmup", warmup + 1, "selesai")

        with ResourceSampler() as idle_sampler:
            time.sleep(IDLE_SAMPLE_SECONDS)
        try:
            resident_models = json_get(OLLAMA_URL + "/api/ps")
        except requests.RequestException:
            resident_models = {"unavailable": True}
        save_json(OUT / "resources" / f"idle_{model}.json",
                  {"summary": idle_sampler.as_dict(), "samples": idle_sampler.samples,
                   "resident_models": resident_models})
        for repeat in range(1, N_REPEAT + 1):
            for question_index, question in enumerate(TEST_QUESTIONS, 1):
                row = make_row(model, question_index, question, repeat)
                sampler, body, fatal, interrupted = ResourceSampler(), None, False, False
                started = None
                try:
                    with sampler:
                        started = time.perf_counter()
                        response = requests.post(BASE_URL + "/api/ask", json=request_payload(question, model),
                                                 timeout=REQUEST_TIMEOUT)
                        try:
                            row["status_code"] = response.status_code
                            row["response_body_bytes"] = len(response.content)
                            request_body = response.request.body
                            row["request_body_bytes"] = len(request_body.encode("utf-8") if isinstance(request_body, str) else (request_body or b""))
                            try:
                                body = response.json()
                            except ValueError:
                                body = {"unreadable_response": True}
                            row["client_total_s"] = time.perf_counter() - started
                            response.raise_for_status()
                        finally:
                            response.close()
                    validate_final_response(body, model)
                    row.update(flatten_response(body))
                    row["client_overhead_s"] = row["client_total_s"] - row["backend_total_s"]
                    row["status"] = "ok"
                except KeyboardInterrupt:
                    row.update(status="interrupted", error_type="KeyboardInterrupt")
                    interrupted = True
                except requests.HTTPError:
                    row["error_type"] = "HTTPError"
                    detail = body.get("detail") if isinstance(body, dict) else None
                    row["error_code"] = detail.get("code") if isinstance(detail, dict) else None
                    fatal = row["status_code"] in (401, 403, 404, 422)
                except (requests.Timeout, requests.ConnectionError) as exc:
                    row["error_type"] = type(exc).__name__
                    fatal = True
                except Exception as exc:
                    row["error_type"] = type(exc).__name__
                    fatal = True
                    print("Pemeriksaan dihentikan:", str(exc))
                if row["status"] != "ok" and started is not None:
                    row["failure_wall_s"] = time.perf_counter() - started
                row.update(sampler.as_dict())
                row["resident_models_before_json"] = json.dumps(resident_models, ensure_ascii=False)
                save_json(OUT / "resources" / f"{model}_r{repeat}_q{question_index}.json",
                          {"samples": sampler.samples, "summary": sampler.as_dict()})
                if body is not None:
                    save_json(OUT / "responses" / f"{model}_r{repeat}_q{question_index}.json",
                              {"question": question, "response": body})
                record_row(row)
                print(model, row["question_id"], "ulangan", repeat, row["status"],
                      "total", round(row.get("client_total_s", row.get("failure_wall_s", 0)), 3), "detik")
                if interrupted:
                    raise KeyboardInterrupt("Hasil parsial tersimpan. Pastikan server sudah berhenti sebelum sesi baru.")
                if fatal:
                    raise RuntimeError("Sesi dihentikan; hasil parsial tersimpan. Periksa error dan backend sebelum mengulang.")
    print("Pengujian selesai:", METRICS_CSV)

run_benchmark()

llama3.2-finetuned warmup 1 selesai
llama3.2-finetuned Q01 ulangan 1 ok total 8.351 detik
llama3.2-finetuned Q02 ulangan 1 ok total 23.675 detik
llama3.2-finetuned Q03 ulangan 1 ok total 32.038 detik
llama3.2-finetuned Q04 ulangan 1 ok total 17.195 detik
llama3.2-finetuned Q01 ulangan 2 ok total 9.635 detik
llama3.2-finetuned Q02 ulangan 2 ok total 18.744 detik
llama3.2-finetuned Q03 ulangan 2 ok total 27.326 detik
llama3.2-finetuned Q04 ulangan 2 ok total 9.289 detik
llama3.2-finetuned Q01 ulangan 3 ok total 9.077 detik
llama3.2-finetuned Q02 ulangan 3 ok total 16.335 detik
llama3.2-finetuned Q03 ulangan 3 ok total 24.941 detik
llama3.2-finetuned Q04 ulangan 3 ok total 8.945 detik
gemma2-finetuned warmup 1 selesai
gemma2-finetuned Q01 ulangan 1 ok total 3.692 detik
gemma2-finetuned Q02 ulangan 1 ok total 7.865 detik
gemma2-finetuned Q03 ulangan 1 ok total 11.201 detik
gemma2-finetuned Q04 ulangan 1 ok total 12.233 detik
gemma2-finetuned Q01 ulangan 2 ok total 4.395 detik
gemma2-finetu

## 5. Ringkasan hasil

Hasil dikelompokkan menurut model, pertanyaan, dan kondisi cache. Waktu retrieval mencakup ekspansi dan pencarian/reranking; waktu total backend sudah mencakup antrean. P95 dari sampel kecil bersifat deskriptif. Keberhasilan permintaan tidak menyatakan jawaban pasti benar.

In [10]:
frame = pd.read_csv(METRICS_CSV, encoding="utf-8-sig")
expected = {(m, f"Q{i:02d}", r) for m in MODEL_KEYS
            for i in range(1, len(TEST_QUESTIONS)+1) for r in range(1, N_REPEAT+1)}
actual = set(zip(frame.model_key, frame.question_id, frame["repeat"]))
coverage = {
    "expected_rows": len(expected), "recorded_unique_rows": len(actual),
    "missing": [list(k) for k in sorted(expected - actual)],
    "unexpected": [list(k) for k in sorted(actual - expected)],
    "duplicate_rows": int(frame.duplicated(["model_key", "question_id", "repeat"]).sum()),
    "ok_rows": int(frame.status.eq("ok").sum()),
    "failed_or_interrupted_rows": int(frame.status.ne("ok").sum()),
    "all_requests_recorded": expected == actual and len(frame) == len(expected),
}
save_json(OUT / "coverage.json", coverage)
display(pd.DataFrame([coverage]))
display(frame.groupby(["model_key", "status", "answer_status"], dropna=False).size().rename("jumlah").reset_index())
valid = frame.loc[frame.status.eq("ok")].copy()

def summarize(data, groups):
    return data.groupby(groups, dropna=False).agg(
        n=("client_total_s", "size"),
        retrieval_mean_s=("retrieval_s", "mean"),
        generation_mean_s=("generation_s", "mean"),
        total_mean_s=("client_total_s", "mean"),
        total_median_s=("client_total_s", "median"),
        total_p95_s=("client_total_s", lambda s: s.quantile(.95)),
        peak_backend_rss_mb=("peak_backend_rss_mb", "max"),
        peak_ollama_rss_mb=("peak_ollama_rss_mb", "max"),
        peak_process_rss_mb=("peak_process_rss_mb", "max"),
        peak_qa_gpu_single_device_mb=("peak_qa_gpu_single_device_mb", "max"),
        qa_cpu_core_seconds_mean=("qa_cpu_core_seconds", "mean"),
        qa_cpu_mean_cores=("qa_cpu_mean_cores", "mean"),
    ).reset_index()

if valid.empty:
    print("Tidak ada respons yang lolos kontrak. Jangan menyimpulkan performa dari permintaan gagal.")
else:
    print("Keseluruhan (campuran cache sesuai pengujian):")
    display(summarize(valid, ["model_key"]))
    print("Dipisahkan menurut cache dan status jawaban:")
    display(summarize(valid, ["model_key", "cache_expansion", "cache_search_rerank", "answer_status"]))
    print("Semua pertanyaan:")
    display(summarize(valid, ["model_key", "question_id"]))
    print("Penghentian generasi dan kelengkapan metrik token:")
    display(valid.groupby(["model_key", "output_finish_reason", "provider_metrics_available"],
                          dropna=False).size().rename("jumlah").reset_index())

print("Satu CSV seluruh metrik:", METRICS_CSV)
print("Metadata lingkungan:", OUT / "system_performance_environment.json")
if not coverage["all_requests_recorded"]:
    print("PERHATIAN: cakupan belum lengkap; hasil ini adalah pengujian parsial.")

,expected_rows,recorded_unique_rows,missing,unexpected,duplicate_rows,ok_rows,failed_or_interrupted_rows,all_requests_recorded
0,24,24,[],[],0,24,0,True


,model_key,status,answer_status,jumlah
0,gemma2-finetuned,ok,basic_checks_passed,3
1,gemma2-finetuned,ok,needs_review,9
2,llama3.2-finetuned,ok,needs_review,12


Keseluruhan (campuran cache sesuai pengujian):


,model_key,n,retrieval_mean_s,generation_mean_s,total_mean_s,total_median_s,total_p95_s,peak_backend_rss_mb,peak_ollama_rss_mb,peak_process_rss_mb,peak_qa_gpu_single_device_mb,qa_cpu_core_seconds_mean,qa_cpu_mean_cores
0,gemma2-finetuned,12,1.257750,4.81025,6.092649,4.901047,11.665060,1523.074219,2443.941406,3944.808594,NaN,10.354167,1.674444
1,llama3.2-finetuned,12,1.633417,15.47325,17.129337,16.765163,29.446713,1908.300781,1790.332031,3529.910156,NaN,51.055990,2.910087


Dipisahkan menurut cache dan status jawaban:


...

Semua pertanyaan:


,model_key,question_id,n,retrieval_mean_s,generation_mean_s,total_mean_s,total_median_s,total_p95_s,peak_backend_rss_mb,peak_ollama_rss_mb,peak_process_rss_mb,peak_qa_gpu_single_device_mb,qa_cpu_core_seconds_mean,qa_cpu_mean_cores
0,gemma2-finetuned,Q01,3,0.003333,4.012000,4.051029,4.066133,4.362080,1523.046875,2443.941406,3944.808594,NaN,5.687500,1.393329
1,gemma2-finetuned,Q02,3,1.021000,4.624000,5.667559,4.593560,7.537790,1523.074219,2423.949219,3924.816406,NaN,9.244792,1.660967
2,gemma2-finetuned,Q03,3,1.914333,5.069000,7.000932,4.938939,10.574527,1523.074219,2409.964844,3910.835938,NaN,12.854167,1.848471
3,gemma2-finetuned,Q04,3,2.092333,5.536000,7.651075,5.379465,11.547294,1508.785156,2422.316406,3923.191406,NaN,13.630208,1.795011
4,llama3.2-finetuned,Q01,3,0.001333,8.999667,9.021092,9.076944,9.579518,1908.300781,1790.308594,3339.171875,NaN,25.239583,2.791191
5,llama3.2-finetuned,Q02,3,2.404667,17.156667,19.584579,18.743827,23.181911,1908.300781,1790.332031,3529.910156,NaN,56.020833,2.891367
6,llama3.2-finetuned,Q03,3,2.257667,25.813667,28.101739,27.326232,31.567193,1494.214844,1788.289062,3282.500000,NaN,92.734375,3.317921
7,llama3.2-finetuned,Q04,3,1.870000,9.923000,11.809940,9.289404,16.404844,1501.332031,1788.335938,3282.550781,NaN,30.229167,2.639871


Penghentian generasi dan kelengkapan metrik token:


,model_key,output_finish_reason,provider_metrics_available,jumlah
0,gemma2-finetuned,model_stop,True,12
1,llama3.2-finetuned,model_stop,True,9
2,llama3.2-finetuned,sentence_boundary,False,3


Satu CSV seluruh metrik: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil\Hasil\tujuan3\versi-final\20261012-200440-99c5f27b\system_performance_all.csv
Metadata lingkungan: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil\Hasil\tujuan3\versi-final\20261012-200440-99c5f27b\system_performance_environment.json


## 6. Pengujian permintaan bersamaan

Uji 1, 2, dan 4 permintaan klien bersamaan, dengan total 72 permintaan terukur di luar pemanasan. Batas proses aktif tetap mengikuti antrean backend. Pertanyaan dipanaskan dan diulang, sehingga hasil menggambarkan beban dengan cache. Profil kelompok disimpan pada CSV terpisah.

In [11]:
from concurrent.futures import ThreadPoolExecutor

# Uji tambahan, terpisah dari 24 baris benchmark. Aktifkan di sel konfigurasi.
# Semua permintaan tetap melalui API RAG produksi; tidak mengubah queue server/cache.
CAPACITY_COLUMNS = ["model_key", "client_concurrency", "request_count", "ok_count",
                    "error_rate", "elapsed_s", "throughput_ok_rps", "response_p95_s"]
CAPACITY_CSV = OUT / "capacity_profile.csv"
def capacity_request(item):
    model, qi, question = item
    started = time.perf_counter()
    result = {"question_id": qi, "model_key": model, "status": "error"}
    body = None
    try:
        with requests.post(BASE_URL + "/api/ask", json=request_payload(question, model),
                           timeout=REQUEST_TIMEOUT) as response:
            result["status_code"] = response.status_code
            result["response_body_bytes"] = len(response.content)
            request_body = response.request.body
            result["request_body_bytes"] = len(request_body.encode("utf-8") if isinstance(request_body, str) else (request_body or b""))
            response.raise_for_status()
            body = response.json()
        validate_final_response(body, model)
        result.update(status="ok", **flatten_response(body))
    except Exception as exc:
        result["error_type"] = type(exc).__name__
    result["client_total_s"] = time.perf_counter() - started
    return result

def run_capacity_profile():
    global CAPACITY_CSV
    if CAPACITY_CSV.exists() or (OUT / "capacity_profile.csv.latest.json").exists():
        raise RuntimeError("Profil sesi sudah ada; gunakan sesi baru, jangan menimpa.")
    records = []
    for model in MODEL_KEYS:
        # Prime empat pertanyaan; profil ini mengukur repeated-query/warm-cache,
        # bukan klaim kapasitas trafik baru tanpa cache.
        for qi, q in enumerate(TEST_QUESTIONS, 1):
            if capacity_request((model, qi, q))["status"] != "ok":
                raise RuntimeError("Pemanasan profil gagal. Pastikan pekerjaan backend selesai sebelum mengulang.")
        for concurrency in CAPACITY_CONCURRENCY_LEVELS:
            try:
                resident = json_get(OLLAMA_URL + "/api/ps")
            except requests.RequestException:
                resident = {"unavailable": True}
            with ResourceSampler() as idle_sampler:
                time.sleep(IDLE_SAMPLE_SECONDS)
            tasks = [(model, i, q) for _ in range(CAPACITY_REPEATS)
                     for i, q in enumerate(TEST_QUESTIONS, 1)]
            with ResourceSampler() as busy_sampler:
                started = time.perf_counter()
                with ThreadPoolExecutor(max_workers=concurrency) as pool:
                    responses = list(pool.map(capacity_request, tasks))
                elapsed = time.perf_counter() - started
            good = [r for r in responses if r["status"] == "ok"]
            resource = busy_sampler.as_dict()
            idle_resource = idle_sampler.as_dict()
            row = {
                **resource, "profile_schema": "qa_capacity_v1", "model_key": model,
                "client_concurrency": concurrency, "request_count": len(tasks), "ok_count": len(good),
                "error_rate": 1-len(good)/len(tasks), "elapsed_s": elapsed,
                "throughput_ok_rps": len(good)/elapsed,
                "response_p95_s": float(pd.Series([r["client_total_s"] for r in good]).quantile(.95)) if good else None,
                "cache_policy": "primed_repeated_questions",
                "observed_all_cache_hits": bool(good) and all(r["cache_expansion"] and r["cache_search_rerank"] for r in good),
                "idle_qa_rss_mb": idle_resource["peak_process_rss_mb"],
                "idle_qa_cpu_cores": idle_resource["qa_cpu_mean_cores"],
                "idle_qa_gpu_single_device_mb": idle_resource["peak_qa_gpu_single_device_mb"],
                "rss_samples_complete": resource["valid_samples_process_rss_mb"] == resource["resource_samples"] and resource["resource_sample_errors"] == 0,
                "resident_models_before_json": json.dumps(resident, ensure_ascii=False),
                "response_style": RESPONSE_STYLE, "top_k": TOP_K,
                "qa_cpu_core_seconds_per_request": None if resource["qa_cpu_core_seconds"] is None else resource["qa_cpu_core_seconds"]/len(tasks),
                "mean_request_body_bytes": float(pd.Series([r.get("request_body_bytes") for r in good], dtype=float).mean()) if good else None,
                "mean_response_body_bytes": float(pd.Series([r.get("response_body_bytes") for r in good], dtype=float).mean()) if good else None,
            }
            records.append(row)
            CAPACITY_CSV = save_csv_resilient(pd.DataFrame(records), CAPACITY_CSV, "capacity_profile.csv")
            save_json(OUT / f"capacity_{model}_c{concurrency}.json", {
                "summary": row, "idle_samples": idle_sampler.samples,
                "busy_samples": busy_sampler.samples, "requests": responses,
                "isolation_note": "Measurement assumes no unrelated clients use this backend/Ollama.",
            })
            print(model, "concurrency klien", concurrency, "selesai:", len(good), "/", len(tasks))
            if len(good) != len(tasks):
                raise RuntimeError("Profil berhenti setelah kelompok gagal; CSV parsial tersimpan. Tunggu backend selesai sebelum mengulang.")
    return pd.DataFrame(records)

if RUN_CAPACITY_PROFILE:
    display(run_capacity_profile())
else:
    print("Profil concurrency belum dijalankan. Aktifkan RUN_CAPACITY_PROFILE=True untuk menghasilkan bukti kapasitas.")

llama3.2-finetuned concurrency klien 1 selesai: 12 / 12
llama3.2-finetuned concurrency klien 2 selesai: 12 / 12
llama3.2-finetuned concurrency klien 4 selesai: 12 / 12
gemma2-finetuned concurrency klien 1 selesai: 12 / 12
gemma2-finetuned concurrency klien 2 selesai: 12 / 12
gemma2-finetuned concurrency klien 4 selesai: 12 / 12


,resource_schema,resource_scope,isolated_run_confirmed,resource_samples,resource_sample_errors,sample_interval_s,memory_unit,baseline_backend_rss_mb,peak_backend_rss_mb,valid_samples_backend_rss_mb,...,idle_qa_rss_mb,idle_qa_cpu_cores,idle_qa_gpu_single_device_mb,rss_samples_complete,resident_models_before_json,response_style,top_k,qa_cpu_core_seconds_per_request,mean_request_body_bytes,mean_response_body_bytes
0,qa_process_v2,backend_and_ollama_unique_process_trees,True,1773,0,0.05,MiB,1500.953125,1500.960938,1773,...,3692.105469,0.030952,None,True,"{""models"": [{""name"": ""llama3.2-finetuned:lates...",detail,5,37.641927,196.25,18509.833333
1,qa_process_v2,backend_and_ollama_unique_process_trees,True,1458,0,0.05,MiB,1485.132812,1511.515625,1458,...,3667.488281,0.015366,None,True,"{""models"": [{""name"": ""llama3.2-finetuned:lates...",detail,5,37.614583,196.25,18512.833333
2,qa_process_v2,backend_and_ollama_unique_process_trees,True,1518,0,0.05,MiB,1511.515625,1537.957031,1518,...,3550.121094,0.007708,None,True,"{""models"": [{""name"": ""llama3.2-finetuned:lates...",detail,5,40.638021,196.25,18515.666667
3,qa_process_v2,backend_and_ollama_unique_process_trees,True,737,0,0.05,MiB,1497.082031,1497.082031,737,...,3649.179688,0.000000,None,True,"{""models"": [{""name"": ""gemma2-finetuned:latest""...",detail,5,9.341146,194.25,18610.833333
4,qa_process_v2,backend_and_ollama_unique_process_trees,True,446,0,0.05,MiB,1451.257812,1451.296875,446,...,3982.816406,0.022554,None,True,"{""models"": [{""name"": ""gemma2-finetuned:latest""...",detail,5,10.013021,194.25,18612.416667
5,qa_process_v2,backend_and_ollama_unique_process_trees,True,380,0,0.05,MiB,1446.035156,1446.128906,380,...,3978.371094,0.000000,None,True,"{""models"": [{""name"": ""gemma2-finetuned:latest""...",detail,5,8.437500,194.25,18613.000000


## 7. Perencanaan adopsi

Perhitungan dipisahkan antara operasional layanan dan pengembangan model. Data berasal dari hasil uji; trafik, margin, dan pertumbuhan ditetapkan sebagai skenario. Data yang belum diukur tetap kosong.

Cakupan penelitian adalah teks naratif publikasi BPS. Dataset buatan LLM tetap memerlukan pemeriksaan isi dan sumber. Bagian ini membaca hasil yang tersedia, tanpa menjalankan pelatihan baru.

In [12]:
import os
import re
import csv
import json
import math
import hashlib
import uuid
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

AD_RESEARCH_ROOT = Path(os.environ.get("QA_RESEARCH_ROOT",
    r"D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil"))
AD_PROJECT_ROOT = Path(os.environ.get("QA_PROJECT_ROOT",
    r"D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\Sistem QA RAG"))
AD_BENCHMARK_CSV = None  # Explicit Path, CSV sesi ini, atau hasil lengkap terbaru (ditampilkan di bawah).
AD_USE_LATEST_COMPLETE_RUN = True
AD_CAPACITY_CSV = None  # None: cari capacity_profile.csv di folder benchmark yang dipilih saja.
AD_MODEL = "llama3.2-finetuned"
AD_CACHE_FILTER = "miss_both"  # all untuk campuran cache; profil concurrency dilaporkan terpisah.

# Target penggunaan adalah ASUMSI yang bisa diganti BPS, bukan hasil ukur laptop.
AD_SCENARIOS = [
    {"scenario": "kecil", "requests_per_day": 100, "active_hours": 8, "peak_factor": 3.0, "target_p95_s": 90, "target_client_concurrency": 1},
    {"scenario": "menengah", "requests_per_day": 1000, "active_hours": 8, "peak_factor": 3.0, "target_p95_s": 90, "target_client_concurrency": 2},
    {"scenario": "besar", "requests_per_day": 10000, "active_hours": 8, "peak_factor": 3.0, "target_p95_s": 90, "target_client_concurrency": 4},
]
AD_UTILIZATION_TARGET = 0.65  # kebijakan utilisasi, bukan metrik pengujian
AD_DAYS_PER_MONTH = 30
AD_MEMORY_MARGIN = 0.0  # isi margin kebijakan secara eksplisit; kebutuhan OS/agen lain tidak diasumsikan.
AD_STORAGE_MARGIN = 0.0
AD_KB_GROWTH_FACTOR = 1.0  # korpus kini; proyeksi pertumbuhan disk diberi label estimasi.
AD_STORAGE_VERSIONS = 1
AD_MAX_ERROR_RATE = 0.01
AD_EXPANSION_INPUT_TOKENS = None  # isi dari log; API sekarang belum mengekspor metrik ini.
AD_EXPANSION_OUTPUT_TOKENS = None
AD_EXPANSION_CACHE_HIT_RATIO = 0.0  # asumsi; 0 untuk seluruh permintaan baru.
AD_ALLOW_LINEAR_MEMORY_EXTRAPOLATION = False  # perlu >=2 level concurrency terukur dengan cache sama.
# Hanya folder/berkas artefak sistem yang nyata; jangan masukkan drive laptop.
AD_ARTIFACT_PATHS = {
    "documents": None,  # Path folder publikasi yang akan disimpan di server
    "embedding_model": None, "reranker_model": None,
    "runtime_environment": None,  # folder environment server yang benar-benar dipakai
    "logs": None,  # snapshot log saat ini; pertumbuhan/retensi perlu pengukuran terpisah
}
AD_TARGET_TRAIN_EXAMPLES = 10000
AD_TARGET_EPOCHS = 3
AD_RUNS_PER_MODEL = 3
AD_SAME_TRAINING_SETUP = True
AD_ALLOCATED_GPUS_PER_RUN = None
AD_EXTRA_PIPELINE_HOURS_PER_RUN = None
AD_DATASET = {
    "accepted_qa_target": 10000, "qa_per_call": 5, "acceptance_rate": 0.80,
    "extra_call_fraction": 0.10, "prompt_tokens_per_call": None,
    "output_tokens_per_call": None, "call_duration_s": None,
    "parallel_calls": 1, "review_sample_fraction": 0.10, "review_minutes_per_qa": 2.0,
}  # semua target ini skenario development, bukan ukuran server yang telah dibuktikan.

AD_SOURCE_FILES = {}
def ad_track(path):
    path = Path(path)
    AD_SOURCE_FILES[str(path)] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path
def ad_positive(value, label, allow_zero=False):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError(label + " harus berupa angka berhingga.")
    if value < 0 or (value == 0 and not allow_zero):
        raise ValueError(label + " di luar rentang.")
    return value
assert AD_CACHE_FILTER in {"miss_both", "all"}
assert 0 < AD_UTILIZATION_TARGET < 1
assert 0 <= AD_EXPANSION_CACHE_HIT_RATIO <= 1
assert 0 <= AD_MEMORY_MARGIN < 1 and 0 <= AD_STORAGE_MARGIN < 1
assert AD_KB_GROWTH_FACTOR >= 1
assert type(AD_STORAGE_VERSIONS) is int and AD_STORAGE_VERSIONS >= 1
print("Model:", AD_MODEL, "| Performa yang belum terukur tetap kosong; tidak ada fallback angka simulasi.")

Model: llama3.2-finetuned | Performa yang belum terukur tetap kosong; tidak ada fallback angka simulasi.


### Kebutuhan layanan

- CPU: laju permintaan × waktu CPU per permintaan ÷ target utilisasi, dalam core-equivalent perangkat uji.
- RAM/VRAM: gunakan peak pada concurrency yang diuji; ekstrapolasi hanya jika diaktifkan dan tersedia beberapa tingkat pengukuran.
- Disk/jaringan: ukuran artefak dan byte respons dikalikan skenario pertumbuhan/trafik.

Token yang tercatat berasal dari jawaban utama; token ekspansi belum tersedia. Estimasi untuk perangkat lain tetap perlu validasi. Hasil uji cache tidak mewakili seluruh pertanyaan baru.

In [13]:
def resolve_latest_csv(path, logical_name):
    """Read the active CSV pointer; never follow a path outside the result folder."""
    path = Path(path)
    if path.name != logical_name:
        return path  # Path hasil pengganti yang dipilih eksplisit harus dihormati.
    pointer = path.parent / (logical_name + ".latest.json")
    if not pointer.is_file():
        return path
    info = json.loads(pointer.read_text(encoding="utf-8"))
    name = info.get("filename")
    stem = Path(logical_name).stem
    if (info.get("logical_name") != logical_name or not isinstance(name, str)
            or Path(name).name != name or "/" in name or "\\" in name
            or not name.endswith(".csv") or not (name == logical_name or name.startswith(stem + "-unlocked-"))):
        raise ValueError("Penunjuk CSV tidak valid: " + str(pointer))
    candidate = path.parent / name
    if not candidate.is_file():
        raise FileNotFoundError("CSV aktif belum ditemukan: " + str(candidate))
    return candidate


def ad_numbers(frame, column):
    if column not in frame:
        return pd.Series(dtype=float)
    return pd.to_numeric(frame[column], errors="coerce").replace([float("inf"), -float("inf")], float("nan")).dropna()

def ad_bool(series):
    return series.astype(str).str.strip().str.lower().eq("true")

def ad_stat(data, column, operation="mean"):
    values = ad_numbers(data, column)
    if (values < 0).any():
        raise ValueError("Metrik negatif: " + column)
    if values.empty:
        return None
    return float(values.quantile(.95) if operation == "p95" else getattr(values, operation)())

def ad_read_profile(path):
    data = pd.read_csv(ad_track(path), encoding="utf-8-sig")
    required = {"test_scope", "model_key", "status", "citation_s", "processing_s",
                "client_total_s", "cache_expansion", "cache_search_rerank", "answer_status"}
    if required - set(data) or not data["test_scope"].eq("e2e_system_final").all():
        raise ValueError("Pilih CSV benchmark sistem versi akhir.")
    model_rows = data.loc[data.model_key.eq(AD_MODEL)]
    if model_rows.empty:
        raise ValueError("Model tidak ditemukan dalam hasil.")
    ok = model_rows.loc[model_rows.status.eq("ok")].copy()
    if pd.to_numeric(ok.citation_s, errors="coerce").isna().any() or pd.to_numeric(ok.citation_s).ne(0).any():
        raise ValueError("CSV masih memiliki generasi ringkasan sitasi/kontrak tidak lengkap.")
    usable = ok if AD_CACHE_FILTER == "all" else ok.loc[
        ok.cache_expansion.astype(str).str.lower().eq("false") &
        ok.cache_search_rerank.astype(str).str.lower().eq("false")]
    p = {"basis": "PENGUKURAN_BENCHMARK", "cache_filter": AD_CACHE_FILTER,
         "model_rows": len(model_rows), "selected_rows": len(usable),
         "failed_rows": int(model_rows.status.ne("ok").sum()),
         "answer_status_counts": usable.answer_status.value_counts().to_dict()}
    for name, column, operation in [
        ("processing_mean_s", "processing_s", "mean"),
        ("client_p95_s", "client_total_s", "p95"),
        ("retrieval_mean_s", "retrieval_s", "mean"),
        ("generation_mean_s", "generation_s", "mean"),
        ("process_ram_peak_mib", "peak_process_rss_mb", "max"),
        ("request_body_bytes_mean", "request_body_bytes", "mean"),
        ("response_body_bytes_mean", "response_body_bytes", "mean"),
    ]:
        p[name] = ad_stat(usable, column, operation)
    scoped = usable.loc[usable.get("resource_schema", pd.Series("", index=usable.index)).eq("qa_process_v2")]
    isolated = scoped.loc[ad_bool(scoped.get("isolated_run_confirmed", pd.Series(False, index=scoped.index)))]
    cpu = isolated.loc[ad_bool(isolated.get("qa_cpu_counter_complete", pd.Series(False, index=isolated.index)))]
    p["cpu_measured_rows"] = len(cpu)
    p["qa_cpu_core_seconds_mean"] = ad_stat(cpu, "qa_cpu_core_seconds")
    p["qa_gpu_peak_per_device_mib"] = ad_stat(isolated, "peak_qa_gpu_single_device_mb", "max")
    tokens = usable.loc[ad_bool(usable.get("provider_metrics_available", pd.Series(False, index=usable.index)))]
    p["token_reported_rows"] = len(tokens)
    for name, col in [("answer_input_tokens", "ollama_prompt_eval_count"), ("answer_output_tokens", "ollama_eval_count")]:
        p[name] = ad_stat(tokens, col)
        p[name + "_n"] = len(ad_numbers(tokens, col))
    p["legacy_host_cpu_gpu_ignored"] = "resource_schema" not in data or not data.resource_schema.eq("qa_process_v2").all()
    return p

ad_candidate = AD_BENCHMARK_CSV
if ad_candidate is None and "METRICS_CSV" in globals() and Path(METRICS_CSV).is_file():
    ad_candidate = Path(METRICS_CSV)
if ad_candidate is None and AD_USE_LATEST_COMPLETE_RUN:
    candidates = []
    for p in (AD_RESEARCH_ROOT / "Hasil/tujuan3/versi-final").glob("*/system_performance_all.csv"):
        coverage_path = p.parent / "coverage.json"
        if coverage_path.is_file():
            coverage = json.loads(coverage_path.read_text(encoding="utf-8"))
            if coverage.get("all_requests_recorded") is True:
                candidates.append(resolve_latest_csv(p, "system_performance_all.csv"))
    if candidates:
        ad_candidate = max(candidates, key=lambda p: p.stat().st_mtime_ns)
if ad_candidate is not None:
    ad_candidate = resolve_latest_csv(Path(ad_candidate), "system_performance_all.csv")
AD_PROFILE = ad_read_profile(Path(ad_candidate)) if ad_candidate is not None else {
    "basis": "BELUM_ADA_PENGUKURAN", "selected_rows": 0}
AD_ENVIRONMENT = {}
if ad_candidate is not None:
    env_path = Path(ad_candidate).parent / "system_performance_environment.json"
    if env_path.is_file():
        AD_ENVIRONMENT = json.loads(ad_track(env_path).read_text(encoding="utf-8"))
print("CSV acuan:", ad_candidate)
print("Lingkungan acuan:", AD_ENVIRONMENT.get("hardware_inventory", "belum tercatat"))
print("Perangkat acuan hanya identitas alat ukur; tidak disalin menjadi spesifikasi server.")

ad_capacity_path = Path(AD_CAPACITY_CSV) if AD_CAPACITY_CSV else (
    Path(ad_candidate).parent / "capacity_profile.csv" if ad_candidate else None)
if ad_capacity_path is not None:
    ad_capacity_path = resolve_latest_csv(ad_capacity_path, "capacity_profile.csv")
AD_CAPACITY = pd.DataFrame()
if ad_capacity_path is not None and ad_capacity_path.is_file():
    AD_CAPACITY = pd.read_csv(ad_track(ad_capacity_path), encoding="utf-8-sig")
    if not {"profile_schema", "resource_schema", "model_key", "client_concurrency",
            "isolated_run_confirmed", "observed_all_cache_hits", "rss_samples_complete",
            "peak_process_rss_mb", "peak_qa_gpu_single_device_mb", "qa_cpu_core_seconds_per_request",
            "error_rate", "throughput_ok_rps", "response_p95_s"} <= set(AD_CAPACITY):
        raise ValueError("Skema capacity_profile.csv tidak sesuai.")
    if not AD_CAPACITY.profile_schema.eq("qa_capacity_v1").all() or not AD_CAPACITY.resource_schema.eq("qa_process_v2").all():
        raise ValueError("Profil kapasitas belum memakai pengukuran proses QA.")
    # Saat memakai CSV eksternal harus satu sesi benchmark/lingkungan; jangan mencampur perangkat.
    if ad_candidate and ad_capacity_path.parent.resolve() != Path(ad_candidate).parent.resolve():
        raise ValueError("Pilih profil kapasitas dari sesi benchmark yang sama.")
    AD_CAPACITY = AD_CAPACITY.loc[AD_CAPACITY.model_key.eq(AD_MODEL)].copy()

def ad_folder_bytes(path):
    path = Path(path)
    if path.is_symlink() or not path.exists():
        return None
    if path.is_file():
        return path.stat().st_size
    total = 0
    for base, dirs, files in os.walk(path, followlinks=False, onerror=lambda error: (_ for _ in ()).throw(error)):
        dirs[:] = [d for d in dirs if not (Path(base)/d).is_symlink() and not getattr(Path(base)/d, "is_junction", lambda: False)()]
        for name in files:
            f = Path(base)/name
            if not f.is_symlink():
                total += f.stat().st_size
    return total

ad_kb_paths = [AD_PROJECT_ROOT/"app/kb_cache/faiss_index.bin", AD_PROJECT_ROOT/"app/kb_cache/chunk_metadata.jsonl"]
AD_KB_GIB = sum(p.stat().st_size for p in ad_kb_paths)/2**30 if all(p.is_file() for p in ad_kb_paths) else None
AD_INVENTORY = [{"component": "index_and_metadata", "bytes": None if AD_KB_GIB is None else AD_KB_GIB*2**30,
                 "basis": "ukuran berkas saat analisis, bukan RAM", "path": "; ".join(str(p) for p in ad_kb_paths)}]
models = AD_ENVIRONMENT.get("model_inventory_before", [])
ad_selected_model = next((m for m in models if str(m.get("name","")).removesuffix(":latest")==AD_MODEL), None) if isinstance(models,list) else None
AD_INVENTORY.append({"component": "llm_model", "bytes": None if ad_selected_model is None else ad_selected_model.get("size"),
                     "basis": "Ollama inventory pada sesi uji; bukan kebutuhan VRAM", "path": "model_inventory_before"})
for name, path in AD_ARTIFACT_PATHS.items():
    AD_INVENTORY.append({"component": name, "bytes": None if path is None else ad_folder_bytes(path),
                         "basis": "belum diukur" if path is None else "ukuran artefak aktual", "path": None if path is None else str(path)})
AD_STORAGE = pd.DataFrame(AD_INVENTORY)
# Hindari menganggap manifest requirements sebagai bukti versi backend yang sedang berjalan.
AD_ENV_FILES = {}
for name in ("requirements.txt", "requirements-notebook.txt"):
    p = AD_PROJECT_ROOT/name
    if p.is_file():
        AD_ENV_FILES[name] = {"sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
                              "declared_dependencies": p.read_text(encoding="utf-8-sig").splitlines(),
                              "running_backend_verified": False}
display(pd.DataFrame([AD_PROFILE]).T.rename(columns={0:"nilai"}))
display(AD_STORAGE)
print("Profil kapasitas tersedia:", len(AD_CAPACITY), "baris. Nilai kosong tidak diganti nol atau spesifikasi tebakan.")

CSV acuan: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil\Hasil\tujuan3\versi-final\20261012-200440-99c5f27b\system_performance_all.csv
Lingkungan acuan: {'purpose': 'identitas alat ukur; BUKAN rekomendasi kapasitas server', 'os': 'Windows-10-10.0.26300-SP0', 'python': '3.11.16', 'cpu': 'Intel64 Family 6 Model 140 Stepping 1, GenuineIntel', 'physical_cores': 4, 'logical_cores': 8, 'total_ram_mib': 16167.3046875, 'nvidia_gpus_sampled': [{'index': 0, 'name': 'NVIDIA GeForce RTX 3050 Laptop GPU', 'total_vram_mib': 4096.0}]}
Perangkat acuan hanya identitas alat ukur; tidak disalin menjadi spesifikasi server.


,nilai
basis,PENGUKURAN_BENCHMARK
cache_filter,miss_both
model_rows,12
selected_rows,3
failed_rows,0
answer_status_counts,{'needs_review': 3}
processing_mean_s,24.288667
client_p95_s,31.202073
retrieval_mean_s,6.529333
generation_mean_s,17.757667


,component,bytes,basis,path
0,index_and_metadata,1.151337e+08,"ukuran berkas saat analisis, bukan RAM",D:\Politeknik Statistika STIS\Pembelajaran\Skr...
1,llm_model,2.019378e+09,Ollama inventory pada sesi uji; bukan kebutuha...,model_inventory_before
2,documents,NaN,belum diukur,NaN
3,embedding_model,NaN,belum diukur,NaN
4,reranker_model,NaN,belum diukur,NaN
5,runtime_environment,NaN,belum diukur,NaN
6,logs,NaN,belum diukur,NaN


Profil kapasitas tersedia: 3 baris. Nilai kosong tidak diganti nol atau spesifikasi tebakan.


In [14]:
def ad_optional_product(value, multiplier):
    return None if value is None else float(value) * multiplier

def ad_memory_projection(data, column, target, allow_extrapolation=False):
    if data.empty or column not in data:
        return None, "BELUM_DIUKUR"
    points = data[["client_concurrency", column]].copy()
    points[column] = pd.to_numeric(points[column], errors="coerce")
    points = points.dropna().groupby("client_concurrency")[column].max().sort_index()
    if (points < 0).any():
        raise ValueError("Memori negatif.")
    exact = points.loc[points.index == target]
    if not exact.empty:
        return float(exact.iloc[0])/1024, "PEAK_PADA_CONCURRENCY_TERUKUR"
    if not allow_extrapolation or len(points) < 2 or target <= points.index.max():
        return None, "CONCURRENCY_TARGET_BELUM_DIUKUR"
    # Estimasi konservatif sederhana, bukan minimum hardware: peak tertinggi +
    # pertambahan paling curam yang teramati pada level concurrency.
    slopes = [max(0, (float(points.iloc[i])-float(points.iloc[i-1])) /
                  (float(points.index[i])-float(points.index[i-1]))) for i in range(1,len(points))]
    result = float(points.max()) + max(slopes) * (target-float(points.index.max()))
    return result/1024, "ESTIMASI_LINEAR_DARI_MULTI_LEVEL_PERLU_VALIDASI"

def ad_plan_operations(profile, scenarios):
    result = []
    qualified = AD_CAPACITY
    if not qualified.empty:
        qualified = qualified.loc[
            ad_bool(qualified.isolated_run_confirmed) &
            ad_bool(qualified.observed_all_cache_hits) &
            qualified.error_rate.eq(0)
        ].copy()
    known_disk = ad_numbers(AD_STORAGE, "bytes").sum()/2**30
    disk_complete = AD_STORAGE["bytes"].notna().all()
    for s in scenarios:
        for key in ("requests_per_day", "active_hours", "peak_factor", "target_p95_s", "target_client_concurrency"):
            ad_positive(s[key], key)
        c = s["target_client_concurrency"]
        if s["active_hours"] > 24 or s["peak_factor"] < 1 or int(c) != c:
            raise ValueError("Jam/faktor puncak/concurrency tidak valid.")
        peak = s["requests_per_day"]/(s["active_hours"]*3600)*s["peak_factor"]
        cpu = profile.get("qa_cpu_core_seconds_mean")
        service = profile.get("processing_mean_s")
        ram_data = qualified.loc[ad_bool(qualified.rss_samples_complete)] if not qualified.empty else qualified
        ram, ram_basis = ad_memory_projection(ram_data, "peak_process_rss_mb", c, AD_ALLOW_LINEAR_MEMORY_EXTRAPOLATION)
        gpu, gpu_basis = ad_memory_projection(qualified, "peak_qa_gpu_single_device_mb", c, AD_ALLOW_LINEAR_MEMORY_EXTRAPOLATION)
        throughput = None
        if not qualified.empty:
            matched = qualified.loc[qualified.client_concurrency.eq(c) & qualified.response_p95_s.le(s["target_p95_s"])]
            throughput = ad_stat(matched, "throughput_ok_rps", "min")
        inp, out = profile.get("answer_input_tokens"), profile.get("answer_output_tokens")
        daily = s["requests_per_day"]
        expansions = daily*(1-AD_EXPANSION_CACHE_HIT_RATIO)
        total_in = None if inp is None or AD_EXPANSION_INPUT_TOKENS is None else daily*inp+expansions*AD_EXPANSION_INPUT_TOKENS
        total_out = None if out is None or AD_EXPANSION_OUTPUT_TOKENS is None else daily*out+expansions*AD_EXPANSION_OUTPUT_TOKENS
        req, resp = profile.get("request_body_bytes_mean"), profile.get("response_body_bytes_mean")
        # Menerapkan pertumbuhan hanya pada footprint indeks saat ini; bukan membesarkan RAM secara otomatis.
        disk_projection = None if not disk_complete else (
            known_disk + (AD_KB_GIB or 0)*(AD_KB_GROWTH_FACTOR-1))*AD_STORAGE_VERSIONS*(1+AD_STORAGE_MARGIN)
        result.append({
            **s, "basis": profile["basis"], "peak_requests_per_s_assumed": peak,
            "cpu_basis": "core-seconds proses QA pada cache_filter benchmark; setara CPU acuan",
            "qa_cpu_core_seconds_per_request_measured": cpu,
            "estimated_cpu_core_equivalents_at_peak": None if cpu is None else cpu*peak/AD_UTILIZATION_TARGET,
            "estimated_independent_service_lanes_NOT_GPU_count": None if service is None else math.ceil(service*peak/AD_UTILIZATION_TARGET),
            "ram_qa_gib_measured_or_estimated": ram, "ram_basis": ram_basis,
            "ram_qa_gib_with_explicit_margin": None if ram is None else ram*(1+AD_MEMORY_MARGIN),
            "gpu_qa_peak_single_device_gib": gpu, "gpu_basis": gpu_basis,
            "gpu_qa_gib_with_explicit_margin": None if gpu is None else gpu*(1+AD_MEMORY_MARGIN),
            "capacity_memory_cache_basis": "primed repeated questions, observed all cache hits",
            "measured_throughput_rps_at_target_concurrency": throughput,
            "reference_replicas_for_measured_cached_workload": None if throughput is None or throughput <= 0 else max(1,math.ceil(peak/(throughput*AD_UTILIZATION_TARGET))),
            "replica_scope": "perkiraan replika identik dengan lingkungan uji; bukan jumlah GPU/server baru",
            "answer_input_tokens_per_day": ad_optional_product(inp,daily),
            "answer_output_tokens_per_day": ad_optional_product(out,daily),
            "total_llm_tokens_per_month": None if total_in is None or total_out is None else (total_in+total_out)*AD_DAYS_PER_MONTH,
            "estimated_api_body_mbps_at_peak_excludes_protocol_and_pdf": None if req is None or resp is None else (req+resp)*peak*8/1e6,
            "known_artifact_disk_gib_subtotal": known_disk,
            "artifact_inventory_complete": bool(disk_complete),
            "projected_artifact_disk_gib_if_complete": disk_projection,
            "server_os_agents_storage_and_ram": "BELUM_DIUKUR_TERPISAH",
        })
    return pd.DataFrame(result)

AD_OPERATIONS = ad_plan_operations(AD_PROFILE, AD_SCENARIOS)
AD_EVIDENCE = pd.DataFrame([
    {"component":"CPU QA", "evidence":"CPU user+system pohon proses per permintaan", "status":"TERUKUR" if AD_PROFILE.get("qa_cpu_core_seconds_mean") is not None else "PERLU_BENCHMARK_PROSES_TERISOLASI", "limit":"core-equivalent CPU acuan; bukan jumlah core CPU lain"},
    {"component":"RAM QA", "evidence":"RSS PID unik backend+Ollama", "status":"TERUKUR_FOOTPRINT" if AD_PROFILE.get("process_ram_peak_mib") is not None else "BELUM_DIUKUR", "limit":"shared pages dapat dihitung lebih dari sekali; bukan minimum RAM fisik"},
    {"component":"VRAM QA", "evidence":"NVML per PID dan per perangkat", "status":"TERUKUR" if AD_PROFILE.get("qa_gpu_peak_per_device_mib") is not None else "BELUM_TERSEDIA", "limit":"WDDM dapat tidak mendukung; tidak memakai total VRAM laptop"},
    {"component":"Concurrency", "evidence":"capacity_profile.csv", "status":"PROFIL_TERSEDIA_PERIKSA_ISOLASI_CACHE" if len(AD_CAPACITY) else "BELUM_DIUJI", "limit":"empat pertanyaan berulang; perlu trafik representatif dan uji panjang"},
    {"component":"Environment", "evidence":"metadata sesi + deklarasi requirements", "status":"TERCATAT_SEBAGIAN", "limit":"versi kernel notebook bukan bukti versi backend; migrasi masih perlu uji integrasi"},
    {"component":"Storage", "evidence":"inventaris artefak sistem", "status":"LENGKAP" if AD_STORAGE.bytes.notna().all() else "INVENTARIS_BELUM_LENGKAP", "limit":"bukan kapasitas disk laptop; retensi log dan ruang ekspor perlu pengukuran"},
    {"component":"Token ekspansi", "evidence":"belum diekspor API", "status":"BELUM_DIUKUR" if AD_EXPANSION_INPUT_TOKENS is None else "INPUT_MANUAL_PERIKSA_LOG", "limit":"jumlah token jawaban bukan total token seluruh pipeline"},
    {"component":"Training penuh", "evidence":"ringkasan trainer + NB08 dry-run", "status":"RUNTIME_ADA_MEMORI_BELUM_LENGKAP", "limit":"optimizer, evaluasi, checkpoint dan ekspor belum diprofilkan penuh"},
])
display(AD_OPERATIONS)
display(AD_EVIDENCE)
print("Estimasi sistem QA tidak memasukkan aplikasi laptop, notebook, OS server, browser, atau agen operasi.")
print("Margin, trafik dan target respons adalah kebijakan/skenario. Resource dasar harus memiliki bukti pengukuran.")

,scenario,requests_per_day,active_hours,peak_factor,target_p95_s,target_client_concurrency,basis,peak_requests_per_s_assumed,cpu_basis,qa_cpu_core_seconds_per_request_measured,...,reference_replicas_for_measured_cached_workload,replica_scope,answer_input_tokens_per_day,answer_output_tokens_per_day,total_llm_tokens_per_month,estimated_api_body_mbps_at_peak_excludes_protocol_and_pdf,known_artifact_disk_gib_subtotal,artifact_inventory_complete,projected_artifact_disk_gib_if_complete,server_os_agents_storage_and_ram
0,kecil,100,8,3.0,90,1,PENGUKURAN_BENCHMARK,0.010417,core-seconds proses QA pada cache_filter bench...,65.53125,...,1.0,perkiraan replika identik dengan lingkungan uj...,219000.0,15350.0,None,0.001586,1.987919,False,None,BELUM_DIUKUR_TERPISAH
1,menengah,1000,8,3.0,90,2,PENGUKURAN_BENCHMARK,0.104167,core-seconds proses QA pada cache_filter bench...,65.53125,...,NaN,perkiraan replika identik dengan lingkungan uj...,2190000.0,153500.0,None,0.015858,1.987919,False,None,BELUM_DIUKUR_TERPISAH
2,besar,10000,8,3.0,90,4,PENGUKURAN_BENCHMARK,1.041667,core-seconds proses QA pada cache_filter bench...,65.53125,...,NaN,perkiraan replika identik dengan lingkungan uj...,21900000.0,1535000.0,None,0.158575,1.987919,False,None,BELUM_DIUKUR_TERPISAH


,component,evidence,status,limit
0,CPU QA,CPU user+system pohon proses per permintaan,TERUKUR,core-equivalent CPU acuan; bukan jumlah core C...
1,RAM QA,RSS PID unik backend+Ollama,TERUKUR_FOOTPRINT,shared pages dapat dihitung lebih dari sekali;...
2,VRAM QA,NVML per PID dan per perangkat,BELUM_TERSEDIA,WDDM dapat tidak mendukung; tidak memakai tota...
3,Concurrency,capacity_profile.csv,PROFIL_TERSEDIA_PERIKSA_ISOLASI_CACHE,empat pertanyaan berulang; perlu trafik repres...
4,Environment,metadata sesi + deklarasi requirements,TERCATAT_SEBAGIAN,versi kernel notebook bukan bukti versi backen...
5,Storage,inventaris artefak sistem,INVENTARIS_BELUM_LENGKAP,bukan kapasitas disk laptop; retensi log dan r...
6,Token ekspansi,belum diekspor API,BELUM_DIUKUR,jumlah token jawaban bukan total token seluruh...
7,Training penuh,ringkasan trainer + NB08 dry-run,RUNTIME_ADA_MEMORI_BELUM_LENGKAP,"optimizer, evaluasi, checkpoint dan ekspor bel..."


Estimasi sistem QA tidak memasukkan aplikasi laptop, notebook, OS server, browser, atau agen operasi.
Margin, trafik dan target respons adalah kebijakan/skenario. Resource dasar harus memiliki bukti pengukuran.


### Pengembangan model

Ringkasan pelatihan dan keluaran NB08/NB09 digunakan sebagai acuan. Waktu target diperkirakan dari jumlah optimizer step dengan perangkat, metode, batch, dan distribusi panjang urutan yang sama.

VRAM NB08 berasal dari satu forward/backward dalam GB desimal, belum mencakup optimizer step atau seluruh proses pelatihan. Jumlah GPU yang terlihat tidak dianggap sebagai jumlah GPU efektif. Token pelatihan diperkirakan dari rata-rata panjang contoh sebelum padding.

In [15]:
def ad_read_json(path):
    return json.loads(ad_track(path).read_text(encoding="utf-8-sig"))

def ad_notebook_output(path):
    book = ad_read_json(path)
    pieces = []
    for cell in book.get("cells", []):
        for output in cell.get("outputs", []):
            value = output.get("text", "")
            pieces.append("".join(value) if isinstance(value, list) else value)
            plain = output.get("data", {}).get("text/plain", "")
            pieces.append("".join(plain) if isinstance(plain, list) else plain)
    text = "\n".join(pieces)
    return re.sub(r"\x1b\[[0-9;]*m", "", text)

def ad_count_csv(path):
    if not path.is_file():
        return None
    ad_track(path)
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return sum(1 for _ in csv.DictReader(handle))

def ad_training_projection(runtime_s, original_steps, examples, epochs, effective_batch, same_setup=True):
    for value, label in ((examples, "examples"), (epochs, "epochs"), (effective_batch, "effective_batch")):
        ad_positive(value, label)
        if int(value) != value:
            raise ValueError(label + " harus bilangan bulat.")
    steps = math.ceil(examples / effective_batch) * epochs
    hours = None
    if same_setup and original_steps is not None and runtime_s is not None:
        ad_positive(original_steps, "original_steps")
        ad_positive(runtime_s, "runtime_s")
        hours = runtime_s / original_steps * steps / 3600
    return steps, hours

ad_training_dir = AD_RESEARCH_ROOT / "Hasil/tujuan2"
ad_config_path = ad_training_dir / "training_config.json"
ad_train_cfg = ad_read_json(ad_config_path) if ad_config_path.is_file() else {}
AD_SPLIT_COUNTS = {name: ad_count_csv(ad_training_dir / (name + ".csv")) for name in ("train", "val", "test")}
ad_preparation = AD_RESEARCH_ROOT / "tujuan2/[2] modelling/08-modelling-preparation.ipynb"
ad_preparation_output = ad_notebook_output(ad_preparation) if ad_preparation.is_file() else ""

ad_rows = []
for key, nb_name in [("gemma2", "09-instruction-tuning-gemma2.ipynb"),
                     ("llama3.2", "09-instruction-tuning-llama3-2.ipynb")]:
    summary_path = ad_training_dir / (key + "_training_summary.json")
    if not summary_path.is_file():
        ad_rows.append({"model": key, "basis": "RINGKASAN_BELUM_TERSEDIA"})
        continue
    summary = ad_read_json(summary_path)
    hp = summary["hyperparameters"]
    cfg = ad_train_cfg.get(key, {})
    runtime = float(summary["total_train_time_sec"])
    train_book = AD_RESEARCH_ROOT / "tujuan2/[2] modelling" / nb_name
    logs = ad_notebook_output(train_book) if train_book.is_file() else ""
    step_match = re.search(r"global_step\s*=\s*(\d+)", logs)
    example_match = re.search(r"Num examples\s*=\s*([\d,]+)", logs)
    original_steps = int(step_match.group(1)) if step_match else None
    original_examples = int(example_match.group(1).replace(",", "")) if example_match else None
    # NB08 text/plain: index, model, token_min, token_mean, token_p95, token_p99, token_max.
    token_match = re.search(r"^\s*\d+\s+" + re.escape(key) +
        r"\s+(\d+)\s+(\d+(?:\.\d+)?)\s+(\d+)\s+(\d+)\s+(\d+)", ad_preparation_output, re.M)
    token_mean = float(token_match.group(2)) if token_match else None
    token_p95 = int(token_match.group(3)) if token_match else None
    token_max = int(token_match.group(5)) if token_match else None
    seq = cfg.get("max_seq_length")
    target_steps, target_hours = ad_training_projection(
        runtime, original_steps, AD_TARGET_TRAIN_EXAMPLES, AD_TARGET_EPOCHS,
        hp["effective_batch_size"], AD_SAME_TRAINING_SETUP)
    dry_gb = cfg.get("vram_estimate_gb", {}).get(str(hp["per_device_batch_size"]))
    dry_gib = None if dry_gb is None else dry_gb * 1e9 / 2**30
    if AD_ALLOCATED_GPUS_PER_RUN is not None:
        ad_positive(AD_ALLOCATED_GPUS_PER_RUN, "allocated GPUs")
        if int(AD_ALLOCATED_GPUS_PER_RUN) != AD_ALLOCATED_GPUS_PER_RUN:
            raise ValueError("Jumlah alokasi GPU harus bulat.")
    if AD_EXTRA_PIPELINE_HOURS_PER_RUN is not None:
        ad_positive(AD_EXTRA_PIPELINE_HOURS_PER_RUN, "extra pipeline hours", allow_zero=True)
    ad_positive(AD_RUNS_PER_MODEL, "runs per model")
    if int(AD_RUNS_PER_MODEL) != AD_RUNS_PER_MODEL:
        raise ValueError("Jumlah eksperimen harus bulat.")
    ad_rows.append({
        "model": key, "basis": "RINGKASAN_TRAINING_DAN_OUTPUT_NOTEBOOK_PENELITIAN",
        "model_name": summary["model_name"], "training_method": "4-bit base + LoRA",
        "lora_rank": hp["lora_r"], "micro_batch": hp["per_device_batch_size"],
        "gradient_accumulation": hp["gradient_accumulation_steps"],
        "effective_batch": hp["effective_batch_size"], "reference_epochs": hp["epochs"],
        "max_sequence_length": seq, "reference_train_examples_from_log": original_examples,
        "current_train_csv_rows": AD_SPLIT_COUNTS["train"],
        "current_train_count_matches_log": None if original_examples is None else original_examples == AD_SPLIT_COUNTS["train"],
        "reference_optimizer_steps": original_steps,
        "measured_trainer_runtime_hours": runtime / 3600,
        "nb08_mean_tokens_rounded": token_mean, "nb08_p95_tokens": token_p95, "nb08_max_tokens": token_max,
        "approx_reference_unpadded_tokens_all_epochs": None if token_mean is None or original_examples is None else token_mean * original_examples * hp["epochs"],
        "dry_run_reserved_gb_decimal_single_device": dry_gb,
        "dry_run_reserved_gib_single_device": dry_gib,
        "dry_run_is_full_training_peak": False,
        "target_train_examples": AD_TARGET_TRAIN_EXAMPLES, "target_epochs": AD_TARGET_EPOCHS,
        "target_optimizer_steps_estimate": target_steps,
        "projected_trainer_hours_same_setup": target_hours,
        "runs_per_model": AD_RUNS_PER_MODEL,
        "projected_serial_experiment_hours": None if target_hours is None else target_hours * AD_RUNS_PER_MODEL,
        "projected_training_gpu_hours_if_allocation_confirmed": None if target_hours is None or AD_ALLOCATED_GPUS_PER_RUN is None else target_hours * AD_RUNS_PER_MODEL * AD_ALLOCATED_GPUS_PER_RUN,
        "projected_full_pipeline_hours": None if target_hours is None or AD_EXTRA_PIPELINE_HOURS_PER_RUN is None else (target_hours + AD_EXTRA_PIPELINE_HOURS_PER_RUN) * AD_RUNS_PER_MODEL,
    })

AD_DEVELOPMENT = pd.DataFrame(ad_rows)
display(AD_DEVELOPMENT)
print("Jumlah baris CSV saat ini:", AD_SPLIT_COUNTS)
print("Jam eksperimen adalah jumlah serial. Paralelisasi memerlukan perangkat tambahan dan pengukuran baru.")
print("GPU-hours di sini hanya fase training; tidak menghitung alokasi saat idle/persiapan/evaluasi terpisah.")

,model,basis,model_name,training_method,lora_rank,micro_batch,gradient_accumulation,effective_batch,reference_epochs,max_sequence_length,...,dry_run_reserved_gib_single_device,dry_run_is_full_training_peak,target_train_examples,target_epochs,target_optimizer_steps_estimate,projected_trainer_hours_same_setup,runs_per_model,projected_serial_experiment_hours,projected_training_gpu_hours_if_allocation_confirmed,projected_full_pipeline_hours
0,gemma2,RINGKASAN_TRAINING_DAN_OUTPUT_NOTEBOOK_PENELITIAN,unsloth/gemma-2-2b-it-bnb-4bit,4-bit base + LoRA,16,4,4,16,3,1024,...,13.326172,False,10000,3,1875,6.127253,3,18.381758,None,None
1,llama3.2,RINGKASAN_TRAINING_DAN_OUTPUT_NOTEBOOK_PENELITIAN,unsloth/Llama-3.2-3B-Instruct-bnb-4bit,4-bit base + LoRA,16,2,8,16,3,1536,...,11.152344,False,10000,3,1875,8.807323,3,26.421968,None,None


Jumlah baris CSV saat ini: {'train': 6516, 'val': 1043, 'test': 1791}
Jam eksperimen adalah jumlah serial. Paralelisasi memerlukan perangkat tambahan dan pengukuran baru.
GPU-hours di sini hanya fase training; tidak menghitung alokasi saat idle/persiapan/evaluasi terpisah.


### Data dan penambahan topik

Publikasi baru melalui ekstraksi, pemeriksaan metadata, chunking, embedding, pembaruan indeks, dan uji regresi. Fine-tuning dilakukan bila evaluasi menunjukkan kebutuhan pembaruan model.

Beban pembuatan dataset dihitung dari target QA layak, jumlah QA per panggilan, tingkat penerimaan, token, dan pengulangan. Waktu pemeriksaan manusia menggunakan asumsi sampling yang dapat disesuaikan.

In [16]:
def ad_dataset_workload(config):
    for key in ("accepted_qa_target", "qa_per_call", "acceptance_rate", "parallel_calls", "review_minutes_per_qa"):
        ad_positive(config[key], key)
    if config["acceptance_rate"] > 1 or not 0 <= config["review_sample_fraction"] <= 1:
        raise ValueError("Proporsi harus berada dalam rentang 0–1.")
    ad_positive(config["extra_call_fraction"], "extra_call_fraction", allow_zero=True)
    for key in ("accepted_qa_target", "qa_per_call", "parallel_calls"):
        if int(config[key]) != config[key]:
            raise ValueError(key + " harus bilangan bulat.")
    calls = math.ceil(config["accepted_qa_target"] / (config["qa_per_call"] * config["acceptance_rate"]))
    calls = math.ceil(calls * (1 + config["extra_call_fraction"]))
    for key in ("prompt_tokens_per_call", "output_tokens_per_call", "call_duration_s"):
        if config[key] is not None:
            ad_positive(config[key], key, allow_zero=key != "call_duration_s")
    serial_h = None if config["call_duration_s"] is None else calls * config["call_duration_s"] / 3600
    reviewed = math.ceil(config["accepted_qa_target"] * config["review_sample_fraction"])
    return {
        "basis": "ASUMSI_PERENCANAAN_DATASET", "accepted_qa_target": config["accepted_qa_target"],
        "estimated_calls_including_extra": calls,
        "estimated_input_tokens": ad_optional_product(config["prompt_tokens_per_call"], calls),
        "estimated_output_tokens": ad_optional_product(config["output_tokens_per_call"], calls),
        "serial_generation_hours": serial_h,
        "ideal_parallel_generation_hours": None if serial_h is None else serial_h / config["parallel_calls"],
        "qa_to_review_under_sampling_assumption": reviewed,
        "review_person_hours": reviewed * config["review_minutes_per_qa"] / 60,
    }

AD_DATASET_PLAN = pd.DataFrame([ad_dataset_workload(AD_DATASET)])
display(AD_DATASET_PLAN)
print("Pembuatan dataset, fine-tuning, dan layanan QA adalah tiga workload berbeda; tokennya tidak dicampur.")

,basis,accepted_qa_target,estimated_calls_including_extra,estimated_input_tokens,estimated_output_tokens,serial_generation_hours,ideal_parallel_generation_hours,qa_to_review_under_sampling_assumption,review_person_hours
0,ASUMSI_PERENCANAAN_DATASET,10000,2750,None,None,None,None,1000,33.333333


Pembuatan dataset, fine-tuning, dan layanan QA adalah tiga workload berbeda; tokennya tidak dicampur.


### RUJUKAN

Rujukan: [efisiensi sumber daya](https://doi.org/10.1145/3706418), [PEFT](https://link.springer.com/article/10.1007/s10462-025-11236-4), [QLoRA](https://proceedings.neurips.cc/paper_files/paper/2023/hash/1feb87871436031bdc0f2beaa62a049b-Abstract-Conference.html), [PagedAttention](https://arxiv.org/abs/2309.06180), dan [Self-Instruct](https://aclanthology.org/2023.acl-long.754/).

In [17]:
AD_REFERENCES = [
    {"title": "Resource-efficient Algorithms and Systems of Foundation Models: A Survey",
     "year": 2025, "venue": "ACM Computing Surveys 57(5), 110", "type": "journal",
     "url": "https://doi.org/10.1145/3706418"},
    {"title": "Parameter-efficient fine-tuning in large language models: a survey of methodologies",
     "year": 2025, "venue": "Artificial Intelligence Review 58, 227", "type": "journal",
     "url": "https://link.springer.com/article/10.1007/s10462-025-11236-4"},
    {"title": "QLoRA: Efficient Finetuning of Quantized LLMs", "year": 2023, "venue": "NeurIPS",
     "type": "conference", "url": "https://proceedings.neurips.cc/paper_files/paper/2023/hash/1feb87871436031bdc0f2beaa62a049b-Abstract-Conference.html"},
    {"title": "Efficient Memory Management for Large Language Model Serving with PagedAttention",
     "year": 2023, "venue": "SOSP", "type": "conference_author_manuscript",
     "url": "https://arxiv.org/abs/2309.06180"},
    {"title": "Self-Instruct: Aligning Language Models with Self-Generated Instructions",
     "year": 2023, "venue": "ACL", "type": "conference",
     "url": "https://aclanthology.org/2023.acl-long.754/"},
]
AD_REFERENCES.extend([
    {"title":"psutil process counters", "type":"official_documentation", "url":"https://psutil.readthedocs.io/stable/index.html"},
    {"title":"NVIDIA NVML process memory and WDDM limitation", "type":"official_documentation",
     "url":"https://docs.nvidia.com/deploy/nvml-api/latest/api/structnvmlProcessDetail__v1__t.html"},
])
AD_ASSUMPTIONS = {
    "model":AD_MODEL, "benchmark_csv":None if ad_candidate is None else str(ad_candidate),
    "capacity_csv":None if ad_capacity_path is None else str(ad_capacity_path),
    "cache_filter":AD_CACHE_FILTER, "scenarios":AD_SCENARIOS,
    "utilization_target":AD_UTILIZATION_TARGET, "days_per_month":AD_DAYS_PER_MONTH,
    "memory_margin":AD_MEMORY_MARGIN, "storage_margin":AD_STORAGE_MARGIN,
    "storage_versions":AD_STORAGE_VERSIONS, "kb_growth_factor":AD_KB_GROWTH_FACTOR,
    "linear_memory_extrapolation_enabled":AD_ALLOW_LINEAR_MEMORY_EXTRAPOLATION,
    "expansion_input_tokens":AD_EXPANSION_INPUT_TOKENS, "expansion_output_tokens":AD_EXPANSION_OUTPUT_TOKENS,
    "expansion_cache_hit_ratio":AD_EXPANSION_CACHE_HIT_RATIO,
    "target_train_examples":AD_TARGET_TRAIN_EXAMPLES, "target_epochs":AD_TARGET_EPOCHS,
    "runs_per_model":AD_RUNS_PER_MODEL, "same_training_setup":AD_SAME_TRAINING_SETUP,
    "allocated_gpus_per_run":AD_ALLOCATED_GPUS_PER_RUN,
    "extra_pipeline_hours_per_run":AD_EXTRA_PIPELINE_HOURS_PER_RUN, "dataset_scenario":AD_DATASET,
}
ad_export_metadata = {
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "method":"process profiling and measurement-based capacity planning",
    "assumptions":AD_ASSUMPTIONS, "operational_profile":AD_PROFILE,
    "reference_environment":AD_ENVIRONMENT,
    "environment_declarations_not_runtime_verification":AD_ENV_FILES,
    "source_sha256":AD_SOURCE_FILES, "references":AD_REFERENCES,
    "dataset_split_counts":AD_SPLIT_COUNTS,
    "limitations":[
        "Missing measurements stay null; host CPU/GPU metrics are not QA resource metrics.",
        "RSS may count shared pages more than once; short peaks/processes may be missed.",
        "QA GPU memory is reported per device, not as pooled VRAM.",
        "Traffic, margins and growth are explicit assumptions; laptop specification is not a server recommendation.",
        "Concurrency profiles use repeated cached questions and do not establish a production SLA.",
        "CPU core-equivalents and reference replica counts apply to the measured hardware/software/workload.",
        "Runtime environment declarations must still be verified against the actual backend and destination server.",
        "Training peak memory with optimizer/checkpoint/export is not measured by the inference notebook.",
    ],
}
ad_metadata_text = json.dumps(ad_export_metadata, ensure_ascii=False, indent=2, allow_nan=False)
AD_OUT = AD_RESEARCH_ROOT / "Hasil/tujuan3/adopsi-bps" / (
    datetime.now().strftime("%Y%m%d-%H%M%S")+"-"+uuid.uuid4().hex[:8])
AD_OUT.mkdir(parents=True, exist_ok=False)
for name,table in [
    ("kebutuhan-operasional.csv",AD_OPERATIONS),
    ("kebutuhan-pengembangan-model.csv",AD_DEVELOPMENT),
    ("beban-pembuatan-dataset.csv",AD_DATASET_PLAN),
    ("inventaris-artefak-sistem.csv",AD_STORAGE),
    ("cakupan-bukti-pengukuran.csv",AD_EVIDENCE),
]:
    table.to_csv(AD_OUT/name,index=False,encoding="utf-8-sig")
(AD_OUT/"asumsi-dan-sumber.json").write_text(ad_metadata_text,encoding="utf-8")
print("Hasil perencanaan berdasarkan bukti:",AD_OUT)

Hasil perencanaan berdasarkan bukti: D:\Politeknik Statistika STIS\Pembelajaran\Skripsi\SourceCode\SkripsiNabil\Hasil\tujuan3\adopsi-bps\20261012-202040-585b91b4
